<a href="https://colab.research.google.com/github/tablitomax-dev/AionUi/blob/main/titanic_trabalho_completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto de Machine Learning Supervisionado: Titanic

**Curso / disciplina:** preencher pelo grupo  
**Integrantes:** preencher pelo grupo  
**Data:** preencher pelo grupo

## Objetivo

Neste notebook, desenvolvemos uma solução de **classificação supervisionada** para prever quais passageiros sobreviveram ao naufrágio do Titanic, utilizando o conjunto de dados disponibilizado pelo Kaggle.[web:12][web:14]

O foco principal deste trabalho não é apenas obter uma pontuação alta, mas sim aplicar corretamente todas as etapas de um projeto clássico de Machine Learning: leitura dos dados, análise exploratória, tratamento, modelagem, avaliação e geração do arquivo de submissão.[web:12]

## 1. Entendimento do problema

O problema consiste em prever a variável-alvo `Survived`, que indica se um passageiro sobreviveu (`1`) ou não (`0`). Isso caracteriza um problema de **classificação binária supervisionada**.

Do ponto de vista metodológico, vamos seguir o fluxo abaixo:

1. Ler as bases `train.csv` e `test.csv`.
2. Realizar análise exploratória dos dados (EDA).
3. Tratar valores ausentes e preparar as variáveis.
4. Treinar e comparar modelos de classificação.
5. Escolher um modelo final.
6. Gerar o arquivo `submission.csv` no formato exigido pelo Kaggle, com `PassengerId` e `Survived`.[web:12][web:14]

In [1]:
# 2. Importação das bibliotecas

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)

## 2. Leitura dos dados

De acordo com o desafio do Kaggle, a base de treino contém a variável `Survived`, enquanto a base de teste é usada para gerar as previsões finais.[web:14]

> **Importante:** antes de executar este notebook, faça o download de `train.csv` e `test.csv` na página da competição e coloque os arquivos na mesma pasta do notebook.[web:12][web:14]

In [15]:
# 3. Leitura das bases

train = pd.read_csv("/content/train.csv", sep=";")
test = pd.read_csv("/content/test.csv", sep=";")

# Remove espaços em branco e joga tudo para minúsculo
train.columns = train.columns.str.strip().str.lower()
test.columns = test.columns.str.strip().str.lower()

print("Dimensão da base de treino:", train.shape)
print("Dimensão da base de teste:", test.shape)
print("\n🔍 Colunas padronizadas:", list(train.columns))

FileNotFoundError: [Errno 2] No such file or directory: '/content/train.csv'

In [16]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


In [ ]:
# Informações gerais

print("Informações da base de treino")
display(train.info())

print("Informações da base de teste")
display(test.info())

## 3. Dicionário resumido das variáveis

As colunas normalmente utilizadas nesse desafio incluem informações demográficas, econômicas e familiares do passageiro, como sexo, idade, classe da passagem, número de irmãos/cônjuges, número de pais/filhos, tarifa e local de embarque.[web:14][web:15]

Variáveis principais deste notebook:

- `Survived`: variável-alvo.
- `Pclass`: classe da passagem.
- `Sex`: sexo do passageiro.
- `Age`: idade.
- `SibSp`: número de irmãos/cônjuges a bordo.
- `Parch`: número de pais/filhos a bordo.
- `Fare`: tarifa paga.
- `Embarked`: porto de embarque.

## 4. Análise exploratória dos dados (EDA)

A análise exploratória é importante para entender a distribuição da variável-alvo, identificar padrões iniciais e detectar problemas como valores ausentes e possíveis inconsistências.[web:13][web:15]

Nesta etapa, vamos observar:

1. Como a variável `Survived` está distribuída.
2. Como a sobrevivência se relaciona com sexo, classe e idade.
3. Quais colunas possuem valores ausentes.

In [ ]:
# 4.1 Estatísticas descritivas

display(train.describe(include="all"))

In [ ]:
# 4.2 Distribuição da variável alvo

col_survived = resolver_coluna(train, "survived")  # funciona com qualquer variação de escrita

plt.figure(figsize=(6, 4))
sns.countplot(data=train, x=col_survived)
plt.title(f"Distribuição da variável alvo: {col_survived}")
plt.xlabel("Sobreviveu")
plt.ylabel("Quantidade")
plt.show()

survival_rate = train[col_survived].mean()
print(f"Taxa média de sobrevivência na base de treino: {survival_rate:.2%}")

In [ ]:
# 4.3 Sobrevivência por sexo

plt.figure(figsize=(6, 4))
sns.countplot(data=train, x="Sex", hue="Survived")
plt.title("Sobrevivência por sexo")
plt.xlabel("Sexo")
plt.ylabel("Quantidade")
plt.show()

sex_survival = train.groupby("Sex")["Survived"].mean().sort_values(ascending=False)
display(sex_survival.to_frame("taxa_sobrevivencia"))

In [ ]:
# 4.4 Sobrevivência por classe

plt.figure(figsize=(6, 4))
sns.countplot(data=train, x="Pclass", hue="Survived")
plt.title("Sobrevivência por classe")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.show()

pclass_survival = train.groupby("Pclass")["Survived"].mean().sort_values(ascending=False)
display(pclass_survival.to_frame("taxa_sobrevivencia"))

In [ ]:
# 4.5 Distribuição da idade por sobrevivência

plt.figure(figsize=(8, 4))
sns.histplot(data=train, x="Age", hue="Survived", kde=True, bins=30, multiple="stack")
plt.title("Distribuição de idade por sobrevivência")
plt.xlabel("Idade")
plt.ylabel("Frequência")
plt.show()

In [ ]:
# 4.6 Distribuição da tarifa

plt.figure(figsize=(8, 4))
sns.boxplot(data=train, x="Survived", y="Fare")
plt.title("Tarifa paga por sobrevivência")
plt.xlabel("Sobreviveu")
plt.ylabel("Fare")
plt.show()

In [ ]:
# 4.7 Valores ausentes

def tabela_missing(df):
    missing = df.isnull().sum()
    percent = df.isnull().mean() * 100
    tabela = pd.DataFrame({
        "missing": missing,
        "percentual": percent
    }).sort_values("percentual", ascending=False)
    return tabela[tabela["missing"] > 0]

print("Valores ausentes - treino")
display(tabela_missing(train))

print("Valores ausentes - teste")
display(tabela_missing(test))

### Comentários da EDA

Com base em análises recorrentes do desafio Titanic, é comum observar muitos valores ausentes em `Cabin`, vários valores ausentes em `Age` e poucos em `Embarked`.[web:13][web:20]

Além disso, `Sex` e `Pclass` costumam apresentar forte relação com a sobrevivência, o que faz sentido histórico e estatístico para esse problema.[web:15][web:16]

> **Sugestão para o relatório:** após executar os gráficos, substituam este texto por interpretações escritas por vocês, comentando o que de fato apareceu nos resultados.

## 5. Preparação dos dados

Nesta etapa, vamos definir quais colunas usar, como tratar valores ausentes e como transformar variáveis categóricas em formato numérico.

### Decisões adotadas

1. Vamos usar as colunas `Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare` e `Embarked`.
2. Vamos remover `Name`, `Ticket` e `Cabin` da abordagem inicial para manter um pipeline didático e objetivo; `Cabin`, em particular, costuma ter muitos valores ausentes.[web:13]
3. Valores ausentes numéricos serão preenchidos com a **mediana**.
4. Valores ausentes categóricos serão preenchidos com a **moda**.
5. Variáveis categóricas serão codificadas com `OneHotEncoder`.

In [ ]:
# 5.1 Seleção de colunas

# Deixamos todos os nomes de colunas em minúsculo aqui também
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target = "survived"

X = train[features].copy()
y = train[target].copy()
X_test = test[features].copy()

num_features = ["age", "sibsp", "parch", "fare", "pclass"]
cat_features = ["sex", "embarked"]

In [ ]:
# 5.2 Separação treino/validação

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

In [ ]:
# 5.3 Pipeline de pré-processamento

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_features),
    ("cat", categorical_transformer, cat_features)
])

## 6. Treinamento e comparação de modelos

O enunciado pede pelo menos um modelo supervisionado, mas recomenda comparar dois ou mais algoritmos. Por isso, vamos testar três modelos clássicos:[web:12]

1. **Regressão Logística**: bom baseline, simples e interpretável.
2. **Árvore de Decisão**: fácil de explicar, mas pode sofrer overfitting.
3. **Random Forest**: costuma melhorar a generalização ao combinar várias árvores.

In [ ]:
# 6.1 Definição dos modelos

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE)
}

In [ ]:
# 6.2 Treinamento e avaliação na validação

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_valid)

    acc = accuracy_score(y_valid, preds)
    f1 = f1_score(y_valid, preds)

    results.append({
        "modelo": name,
        "accuracy": acc,
        "f1_score": f1
    })

    fitted_pipelines[name] = pipeline

results_df = pd.DataFrame(results).sort_values(by="f1_score", ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
# 6.3 Validação cruzada opcional para reforçar a análise

cv_results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(pipeline, X, y, cv=5, scoring="accuracy")
    cv_results.append({
        "modelo": name,
        "cv_accuracy_media": scores.mean(),
        "cv_accuracy_std": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(by="cv_accuracy_media", ascending=False).reset_index(drop=True)
display(cv_results_df)

## 7. Escolha do melhor modelo

Neste ponto, o grupo deve escolher o modelo final com base nas métricas e na capacidade de justificar a escolha. Em muitos casos, o Random Forest apresenta desempenho competitivo nesse desafio, mas essa decisão deve ser confirmada pelos resultados obtidos na execução do notebook.[web:16]

Como regra didática:

- Se dois modelos tiverem desempenho muito parecido, pode ser melhor escolher o mais simples de explicar.
- Se um modelo tiver vantagem consistente em `accuracy` e `f1_score`, ele pode ser adotado como modelo final.

In [ ]:
# 7.1 Seleção automática do melhor modelo pelo F1-score

best_model_name = results_df.loc[0, "modelo"]
best_pipeline = fitted_pipelines[best_model_name]

print("Melhor modelo com base no F1-score da validação:", best_model_name)

In [ ]:
# 7.2 Relatório de classificação

valid_preds = best_pipeline.predict(X_valid)

print(classification_report(y_valid, valid_preds))

In [ ]:
# 7.3 Matriz de confusão

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_valid, valid_preds, ax=ax, cmap="Blues")
plt.title(f"Matriz de confusão - {best_model_name}")
plt.show()

## 8. Importância das variáveis

O enunciado recomenda, quando possível, avaliar a importância das variáveis. Isso é especialmente útil quando o modelo final for baseado em árvores, como Decision Tree ou Random Forest.[web:12]

Se o melhor modelo não for baseado em árvores, esta seção pode ser mantida como análise complementar com o Random Forest.

In [ ]:
# 8.1 Importância das variáveis para modelos baseados em árvore

rf_pipeline = fitted_pipelines["Random Forest"]
rf_model = rf_pipeline.named_steps["model"]
rf_preprocessor = rf_pipeline.named_steps["preprocessor"]

encoded_cat_names = rf_preprocessor.named_transformers_["cat"]    .named_steps["onehot"]    .get_feature_names_out(cat_features)

all_feature_names = num_features + list(encoded_cat_names)

feature_importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

display(feature_importance_df.head(15))

In [ ]:
# 8.2 Gráfico das importâncias

plt.figure(figsize=(8, 5))
sns.barplot(data=feature_importance_df.head(10), x="importance", y="feature")
plt.title("Top 10 variáveis mais importantes - Random Forest")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.show()

## 9. Treinamento final com toda a base de treino

Depois de escolher o modelo final, o passo correto é treiná-lo novamente usando toda a base `train.csv`, para então gerar as previsões da base `test.csv`.[web:14][web:15]

In [ ]:
# 9.1 Definição do modelo final

final_model_lookup = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE)
}

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", final_model_lookup[best_model_name])
])

final_pipeline.fit(X, y)

In [ ]:
# 9.2 Previsões na base de teste

test_preds = final_pipeline.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_preds.astype(int)
})

display(submission.head())
print("Formato do arquivo de submissão:", submission.shape)

In [ ]:
# 9.3 Geração do arquivo final

submission.to_csv("submission.csv", index=False)
print("Arquivo 'submission.csv' gerado com sucesso.")

## 10. Submissão no Kaggle

Segundo a página da competição, o arquivo de submissão deve ser um CSV com cabeçalho e exatamente **418 previsões** para a base de teste, além de seguir o formato esperado pelo desafio.[web:12]

Passos para submissão:

1. Acessar a página da competição Titanic no Kaggle.
2. Clicar em **Submit Predictions**.
3. Enviar o arquivo `submission.csv` gerado neste notebook.
4. Registrar uma captura de tela com o score obtido, pois isso faz parte dos entregáveis do trabalho.[web:12]

## 11. Conclusão

Neste projeto, aplicamos um fluxo completo de Machine Learning supervisionado para um problema de classificação binária. O trabalho incluiu leitura dos dados, EDA, tratamento de ausências, transformação de variáveis, comparação de modelos, avaliação e geração do arquivo final de submissão.[web:12][web:14]

### Pontos que podem ser destacados pelo grupo na versão final

- Quais variáveis pareceram mais relevantes para a sobrevivência.
- Qual modelo apresentou melhor equilíbrio entre simplicidade e desempenho.
- Quais limitações ainda existem na solução.
- Quais melhorias futuras poderiam ser testadas, como engenharia de atributos com `Title`, `FamilySize` ou uso mais elaborado de `Cabin`.[web:20]

## 12. Checklist final da entrega

Antes de entregar, confirmem:

- [ ] O notebook executa do início ao fim sem erros.
- [ ] As células estão em ordem lógica.
- [ ] O texto explica as decisões mais importantes.
- [ ] O arquivo `submission.csv` foi gerado.
- [ ] A submissão no Kaggle foi realizada.
- [ ] Foi salva a captura de tela com o resultado da competição.